## **config.py**

Every setting the pipeline depends on lives in one file. Nothing else in the
codebase hardcodes a path, a class name, or a threshold, they all read from
here.

That's worth doing for two reasons. Tuning the system means editing one file
instead of hunting through seven. And when you change a threshold and the
results shift, you know exactly what you changed.

## **Setup**

Clone the repo so we can look at the real file.

In [5]:
!git clone -q https://github.com/shreyamali17/helmet-violation-detection.git
%cd helmet-violation-detection/project/pipeline

/content/helmet-violation-detection/project/pipeline/helmet-violation-detection/project/pipeline


In [7]:
import os
import config

print("Project root:", config.PROJECT_ROOT)
print("Model:       ", config.MODEL_PATH)
print("Video:       ", config.VIDEO_PATH)
print()
print("model exists:", os.path.exists(config.MODEL_PATH))
print("video exists:", os.path.exists(config.VIDEO_PATH))

Project root: /content/helmet-violation-detection/project/pipeline/helmet-violation-detection/project
Model:        /content/helmet-violation-detection/project/pipeline/helmet-violation-detection/project/models/best.pt
Video:        /content/helmet-violation-detection/project/pipeline/helmet-violation-detection/project/sample_videos/sample.mp4

model exists: True
video exists: True


## **What the Model Knows**

The detection model was trained to recognize **five classes**. Each class is represented by a fixed class ID in the trained model. The pipeline uses this predefined class mapping to correctly interpret the model's detections, such as identifying whether a detection corresponds to a **motorcycle, rider, helmet, no-helmet, or number plate**.

In [8]:
CLASS_NAMES = {
    0: "motorcycle",
    1: "rider",
    2: "helmet",
    3: "no_helmet",
    4: "license_plate",
}

for k, v in config.CLASS_NAMES.items():
    print(f"  {k} -> {v}")

  0 -> motorcycle
  1 -> rider
  2 -> helmet
  3 -> no_helmet
  4 -> license_plate


## **How much video to process ?**

`FRAME_LIMIT` caps how many frames the pipeline reads. `None` means process everything.

This exists for iteration speed. A full 60-second clip at 25 fps is 1500
frames, and every frame runs a neural network. When you're testing a change to the association logic, waiting three minutes to see the result is the difference between trying five ideas and trying one.

`TRACKER` selects the algorithm that assigns stable IDs across frames.
ByteTrack is a deliberate choice for edge hardware : it's lightweight, and heavier trackers would eat the frame budget we're trying to protect.

In [9]:
FRAME_LIMIT = None       # None = process the entire video
TRACKER = "bytetrack.yaml"

print("Frame limit:", config.FRAME_LIMIT)
print("Tracker:    ", config.TRACKER)

Frame limit: None
Tracker:     bytetrack.yaml


## **Matching thresholds**

These three control how willing the system is to pair two objects together. Each is a **maximum cost** : a pairing is rejected if matching the two objects costs more than this.

`RIDER_MOTO_MAX_COST = 0.9` is the important one, and it reads strangely until you know the cost formula. Rider-to-motorcycle cost is `1 - IoU`, where IoU measures how much two boxes overlap. So a cost ceiling of 0.9 means an overlap floor of 0.1 — the boxes must overlap by at least 10% or the pairing is thrown out.

The other two use pixel distance rather than overlap, which is why they're in the tens and hundreds rather than between 0 and 1. A helmet detection more than 60 pixels from a rider's head position isn't that rider's helmet.

In [10]:
RIDER_MOTO_MAX_COST = 0.9   # cost = 1 - IoU, so this requires >= 0.1 overlap
HEAD_RIDER_MAX_COST = 60    # pixels
PLATE_MOTO_MAX_COST = 100   # pixels

print(f"Rider-motorcycle: cost <= {config.RIDER_MOTO_MAX_COST}"
      f"  (IoU >= {1 - config.RIDER_MOTO_MAX_COST:.1f})")
print(f"Head-rider:       <= {config.HEAD_RIDER_MAX_COST} px")
print(f"Plate-motorcycle: <= {config.PLATE_MOTO_MAX_COST} px")

Rider-motorcycle: cost <= 0.9  (IoU >= 0.1)
Head-rider:       <= 60 px
Plate-motorcycle: <= 100 px


## **Deciding a violation**

A single frame is never enough. A rider might be blurred, half-occluded, or caught mid-turn. These two settings are what stop one bad frame from
generating a ticket.

`MIN_OBSERVATIONS = 3` : the rider must have been seen and classified at least three times before any decision is made.

`MIN_CONFIDENCE = 0.7` : of those observations, at least 70% must agree. A rider seen ten times with six `no_helmet` and four `helmet` is a 60% majority, which isn't enough. The system returns "unknown" rather than guessing.

In [11]:
MIN_OBSERVATIONS = 3
MIN_CONFIDENCE = 0.7

# what these two rules do to some example observation sets
examples = [
    ["no_helmet"],
    ["no_helmet", "no_helmet"],
    ["no_helmet", "no_helmet", "no_helmet"],
    ["no_helmet", "no_helmet", "helmet", "helmet", "no_helmet"],
    ["no_helmet", "helmet", "no_helmet", "helmet", "no_helmet"],
]

for votes in examples:
    n = len(votes)
    top = max(set(votes), key=votes.count)
    conf = votes.count(top) / n
    if n < config.MIN_OBSERVATIONS:
        verdict = f"too few observations ({n} < {config.MIN_OBSERVATIONS})"
    elif conf < config.MIN_CONFIDENCE:
        verdict = f"not confident enough ({conf:.0%} < {config.MIN_CONFIDENCE:.0%})"
    else:
        verdict = f"DECIDED: {top} at {conf:.0%}"
    print(f"{n} obs, {conf:.0%} agree -> {verdict}")

1 obs, 100% agree -> too few observations (1 < 3)
2 obs, 100% agree -> too few observations (2 < 3)
3 obs, 100% agree -> DECIDED: no_helmet at 100%
5 obs, 60% agree -> not confident enough (60% < 70%)
5 obs, 60% agree -> not confident enough (60% < 70%)


## **Try it**

Change a threshold and watch what happens downstream. Some things worth
testing:

- Set `RIDER_MOTO_MAX_COST` to `1.0`. Now any overlap at all is acceptable, including zero. How many more pairings appear, and are they right?
- Drop `MIN_OBSERVATIONS` to `1`. Violations will be confirmed from single frames. Do the extra ones hold up?
- Raise `MIN_CONFIDENCE` to `0.9`. The system gets stricter and reports fewer violations. Which ones does it now refuse to call?

There's no single correct setting. The right values depend on whether you'd rather miss a violation or accuse the wrong person.